# Session 10

[![Open and Execute in Google Colaboratory](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrojuanlu/ie-mbd-python-data-analysis-i/blob/main/sessions/Session%2010.ipynb)

- Reading semi-structured data into pandas
- String methods on pandas columns
- The "group by and aggregate" operation
- The `agg` method

## Reading semi-structured data into pandas

pandas DataFrames are always table-like objects, but that doesn't mean that they're limited to CSV data. In fact, you can read many other formats using different `pandas.read_*` methods.

For example, it's possible to read semi-structured data into a pandas DataFrame. Let's do an example with JSON:

In [1]:
# BLUESKY_DATA_URL = "../data/bluesky_more_5000_likes_filtered.json"
BLUESKY_DATA_URL = (
    "https://github.com/astrojuanlu/ie-mbd-python-data-analysis-i/"
    "raw/main/data/bluesky_more_5000_likes_filtered.json"
)

In [2]:
import pandas as pd

In [3]:
df = pd.read_json(BLUESKY_DATA_URL)
df.head(5)

,post_id,user_id,instance,date,text,langs,like_count,reply_count,repost_count,reply_to,replied_author,thread_root,thread_root_author,repost_from,reposted_author,quotes,quoted_author,labels,sent_label,sent_score
0,193115731,2648,ilovecitr.us,1976-05-31 15:05:10.052,Wins for Oppenheimer and Godzilla Minus One me...,[eng],6300,50,2138,NaN,NaN,NaN,NaN,65676,20990,NaN,NaN,NaN,2.0,0.807
1,193116187,2648,ilovecitr.us,1976-05-31 14:34:41.234,Today’s bake for brother’s 40th. I believe I h...,[eng],8266,171,1151,NaN,NaN,NaN,NaN,87314,46762,NaN,NaN,NaN,2.0,0.967
2,193116350,2648,ilovecitr.us,1976-05-31 14:31:01.544,Petition to honor Michael Shannon every Jan. 6,[eng],5327,87,1564,NaN,NaN,NaN,NaN,67433,78046,NaN,NaN,NaN,1.0,0.773
3,193116579,2648,ilovecitr.us,1976-05-30 13:35:00.156,Is it possible to fly a flag at twice-staff?,[eng],5566,55,1470,NaN,NaN,NaN,NaN,68732,64505,NaN,NaN,NaN,1.0,0.910
4,193116658,2648,ilovecitr.us,1976-05-30 13:33:00.024,1923: The world's richest man is an antisemite...,[eng],6719,87,2165,NaN,NaN,NaN,NaN,252171,33509,NaN,NaN,NaN,0.0,0.625


In [4]:
df.dtypes

,0
post_id,int64
user_id,int64
instance,object
date,datetime64[ns]
text,object
langs,object
like_count,int64
reply_count,int64
repost_count,int64
reply_to,float64


In [ ]:
df = pd.read_json(BLUESKY_DATA_URL)
df.head()

Notice several things:

(1) both the `text` and `langs` columns have dtype `object`, even though the former is made of strings and the latter is made of lists!

In [ ]:
df.dtypes

(2) If a given field is present in at least one record, the records that don't have it will hold a `NaN` value. You will learn more about handling missing data in the next session.

---

Alternatively, you could also read this data using the `requests` package into a native Python object:

In [5]:
# import json
import requests

# data = json.load(open(BLUESKY_DATA_URL))
data = requests.get(BLUESKY_DATA_URL).json()
type(data), len(data)

(list, 335)

And then use one of the `pandas.DataFrame.from_*` methods (notice that these are on a different namespace than the `pandas.read_*` methods):

In [ ]:
pd.DataFrame.from_records(data).head(2)

## Exercises

### 1. JSON reading

The `rick-and-morty.json` data is not so easy to read directly as JSON. Go ahead and try it. What's the error?

Use the alternative method: load the `rick-and-morty.json` data to a Python object, then store the episodes list in a variable `episodes`, then pass it to the method `pandas.DataFrame.from_records` to turn the list of episodes into a `DataFrame`.

List the first 5 rows to verify that it was correctly loaded. **Notice that some columns contain dictionaries**.

In [10]:
import requests

RICK_MORTY_DATA_URL = (
    "https://github.com/astrojuanlu/ie-mbd-python-data-analysis-i/"
    "raw/main/data/rick-and-morty.json"
)

rm_data = requests.get(RICK_MORTY_DATA_URL).json()
print(type(rm_data), len(rm_data))

<class 'dict'> 24


In [7]:
df_rm = pd.read_json(RICK_MORTY_DATA_URL)

ValueError: Mixing dicts with non-Series may lead to ambiguous ordering.

In [16]:
rm_data = requests.get(RICK_MORTY_DATA_URL).json()

In [17]:
rm_data


{'id': 216,
 'url': 'https://www.tvmaze.com/shows/216/rick-and-morty',
 'name': 'Rick and Morty',
 'type': 'Animation',
 'language': 'English',
 'genres': ['Comedy', 'Adventure', 'Science-Fiction'],
 'status': 'Running',
 'runtime': 30,
 'averageRuntime': 30,
 'premiered': '2013-12-02',
 'ended': None,
 'officialSite': 'http://www.adultswim.com/videos/rick-and-morty',
 'schedule': {'time': '23:00', 'days': ['Sunday']},
 'rating': {'average': 9},
 'weight': 99,
 'network': {'id': 10,
  'name': 'Adult Swim',
  'country': {'name': 'United States',
   'code': 'US',
   'timezone': 'America/New_York'}},
 'webChannel': None,
 'dvdCountry': None,
 'externals': {'tvrage': 33381, 'thetvdb': 275274, 'imdb': 'tt2861424'},
 'image': {'medium': 'https://static.tvmaze.com/uploads/images/medium_portrait/1/3603.jpg',
  'original': 'https://static.tvmaze.com/uploads/images/original_untouched/1/3603.jpg'},
 'summary': '<p>Rick is a mentally gifted, but sociopathic and alcoholic scientist and a grandfathe

In [19]:
episodes = rm_data["_embedded"]["episodes"]
rm_df = pd.DataFrame.from_records(episodes)
rm_df.head(5)

,id,url,name,season,number,type,airdate,airtime,airstamp,runtime,rating,image,summary,_links
0,14308,https://www.tvmaze.com/episodes/14308/rick-and...,Pilot,1,1,regular,2013-12-02,22:30,2013-12-03T03:30:00+00:00,30,{'average': 8.6},{'medium': 'https://static.tvmaze.com/uploads/...,<p>Rick takes Morty to another dimension to ge...,{'self': {'href': 'https://api.tvmaze.com/epis...
1,14309,https://www.tvmaze.com/episodes/14309/rick-and...,Lawnmower Dog,1,2,regular,2013-12-09,22:30,2013-12-10T03:30:00+00:00,30,{'average': 8.9},{'medium': 'https://static.tvmaze.com/uploads/...,"<p>Morty's small, white dog Snuffles gets on t...",{'self': {'href': 'https://api.tvmaze.com/epis...
2,14310,https://www.tvmaze.com/episodes/14310/rick-and...,Anatomy Park,1,3,regular,2013-12-16,22:30,2013-12-17T03:30:00+00:00,30,{'average': 9},{'medium': 'https://static.tvmaze.com/uploads/...,<p>It's around Christmas time and Jerry's pare...,{'self': {'href': 'https://api.tvmaze.com/epis...
3,14311,https://www.tvmaze.com/episodes/14311/rick-and...,M. Night Shaym-Aliens!,1,4,regular,2014-01-13,22:30,2014-01-14T03:30:00+00:00,30,{'average': 9},{'medium': 'https://static.tvmaze.com/uploads/...,<p>Rick and Morty try to get to the bottom of ...,{'self': {'href': 'https://api.tvmaze.com/epis...
4,14312,https://www.tvmaze.com/episodes/14312/rick-and...,Meeseeks and Destroy,1,5,regular,2014-01-20,22:30,2014-01-21T03:30:00+00:00,30,{'average': 9},{'medium': 'https://static.tvmaze.com/uploads/...,<p>Rick provides the family with a solution to...,{'self': {'href': 'https://api.tvmaze.com/epis...


In [ ]:
import pandas as pd

In [ ]:
rm_df = pd.DataFrame.from_records(data["_embedded"]["episodes"])
rm_df.head()

## The "group by and aggregate" operation

Group by operations in pandas are essential to perform advanced aggregations. The concept and the syntax are directly borrowed from SQL, and follow a similar "split-apply-combine" procedure. At a very high level, this is what happens:

![Group by and aggregate](../img/group-by-agg.png)

Group by operations are initiated by calling the `groupby` method of a DataFrame. But notice that this returns an intermediate object:

In [20]:
df.head(1)

,post_id,user_id,instance,date,text,langs,like_count,reply_count,repost_count,reply_to,replied_author,thread_root,thread_root_author,repost_from,reposted_author,quotes,quoted_author,labels,sent_label,sent_score
0,193115731,2648,ilovecitr.us,1976-05-31 15:05:10.052,Wins for Oppenheimer and Godzilla Minus One me...,[eng],6300,50,2138,NaN,NaN,NaN,NaN,65676,20990,NaN,NaN,NaN,2.0,0.807


In [21]:
df.groupby("instance")

To effectively use this object, you have to finalize the operation by calling an aggregation. The result will be another pandas object, with the index containing each of the distinct values of the column you are grouping by.

For example, to know how many posts are there per instance:

In [22]:
df.groupby("instance").size()

,0
instance,
bsky.social,305
com,4
icopartners.com,3
ilovecitr.us,9
io,1
net,13


And to extract the first (earliest) post of each instance:

In [ ]:
df.groupby("instance").first()

Some aggregations work with specific data types. For example, yhou might be interested in average statistics of some numerical columns:

In [ ]:
(
    df.loc[:, ["instance", "like_count", "reply_count"]]
    .groupby("instance")
    .mean()
)

## Exercises

### 2. Analyzing semi-structured data

Answer the remaining questions about the Rick & Morty data from session 6, using exclusively pandas methods (no comprehensions or loops).

## The `agg` method

Sometimes you want to apply more complex aggregations, or stack several of them, or apply different aggregations to different columns. The `.agg` method of the `DataFrameGroupBy` allows you to do all that.

In [ ]:
(
    df.loc[:, ["instance", "like_count", "reply_count"]]
    .groupby("instance")
    .agg(["mean", "std"])  # A list of aggregation functions
)

In [ ]:
(
    df.loc[:, ["instance", "like_count", "reply_count"]]
    .groupby("instance")
    .agg({"like_count": "mean", "reply_count": ["sum", "std"]})  # A dictionary/mapping of column names to aggregation functions
)

## Exercises

### 3. More analysis of semi-structured data

Load the Reddit data from Session 06 and answer the questions there, using exclusively pandas methods (no comprehensions or loops).

In [23]:
REDDIT_DATA_URL = (
    "https://github.com/astrojuanlu/ie-mbd-python-data-analysis-i/"
    "raw/main/data/reddit_popular.json"
)